# Signal & Background Distributions -- DATA, Before Cosmic Tagger, INPUT: ROOT FILES -- ON-BEAM

The ON-BEAM counterpart of `SignalBackground_Distributions_BeforeCosmicTagger_Data_OffBeam_InputRootFiles.ipynb`
-- same code, pointed at the on-beam (beam-on) sample. The ROOT-file counterpart of
`SignalBackground_Distributions_BeforeCosmicTagger_Data_OnBeam.ipynb`: same
sample (on-beam / beam-on, real data, no truth) and the same reco
selection, but every quantity is read from the per-event
`tracking-pr_r<run>_s<subrun>_e<event>.root` files (`read_tracking_pr_root`,
readfiles.py, via uproot) instead of the staged `clustering-global` / `op` JSONs.
Nothing is staged or unzipped.

**How the JSON selection maps onto `T_cluster`** (checked event by event on
the OFF-BEAM `chunk_00/subchunk_00` against the JSON notebook's own reader):

| JSON notebook | ROOT here |
|---|---|
| a reco cluster = `clustering-global` points grouped by `cluster_id` | a `T_cluster` row with `in_scope == 1 && is_main == 1` (same ids; the `is_associated` rows are folded into their main cluster in the JSON) |
| bridged flash time from `op.json` | `flash_time_us` (agrees to 0.01 us) |
| a cluster with no bridged flash is cut | `flash_id < 0` is cut |
| -- | `T_tagger` rows: the neutrino candidates, a further stage the JSON notebook does not have |

`beam_flash` is 0 even for clusters whose flash is in the beam window, so it is
not used.

**Crediting `T_tagger` candidates.** WireCell's candidate is not always the
cluster the NoCuts stage keeps. A beam-window main cluster passes `+ Nu candidate`
when a candidate is that cluster (`match = main`) or is an associated cluster
sharing its flash (`match = associated`; the JSON folds it into that main cluster).
Any other candidate -- in off-beam chunk_00 these are out-of-scope main clusters, a few cm
long with empty kinematics -- is not in the NoCuts population: it is listed in
`summary.txt` and `selected_clusters.txt` but kept out of the flow and the plots.

**Reco energy.** The ROOT files carry no per-cluster charge, so the JSON
notebook's x-axis (`23.6 eV * charge / recombination`) is built from the staged
`clustering-global` JSON: the selected cluster's points (same `cluster_id` as
`T_cluster`) are summed, and drawn with the JSON notebook's own
`draw_selected_reco_energy` at 200/100/50 MeV for `AfterBeamWindowCut` and
`NuCandidate`. This is the ONLY JSON the notebook reads; a cluster whose event
has no staged JSON is counted in the summary but not drawn.

| population | variables drawn |
|---|---|
| `AfterBeamWindowCut`, `NuCandidate` | reco cluster energy at 200/100/50 MeV (charge from JSON) |
| `AfterBeamWindowCut` clusters | `length_cm`, `npoints`, `flash_pe`, `flash_time_us` |
| `NoCuts` clusters with a flash, -10..10 us | `flash_time_us` zoomed out, beam window shaded (`..._zoomout_-10to10us`) |
| `AfterBeamWindowCut`, every T_tagger row with its flash in the beam window (credited or not) | `numu_score`, `nue_score` |
| `NuCandidate` (credited T_tagger candidates) | `kine_reco_Enu` at 200/100/50 MeV (only where filled, i.e. > 0), `numu_score`, `nue_score` |

All drawn as data points (`draw_selected_data_points`, draw_selection_performance.py):
black markers at bin centres, `sqrt(N)` bars, empty bins undrawn. The BDT score
plots have no error bars. The scores exist only for T_tagger rows: a
beam-window cluster with no T_tagger row has no score to plot.

**BEE.** Every run uploads one BEE set per population (`bee_set_AfterBeamWindowCut`,
`bee_set_NuCandidate`, and `bee_set_AfterBeamWindowCut_NotNuCandidate` for the
beam-window clusters no candidate is credited to; all inside the run directory)
built from the staged zip JSONs,
and writes the set url plus one url per event to
`job_summary/bee_links_<population>.txt`.

**Input layout** (chunk_r3_rootfiles.py): a ROOT file whose bee zip exists sits
next to it in `bee/chunk_NN/subchunk_MM/` (also `after_1000files/`, `corrupted/`);
ROOT files without a zip are in `tracking-pr/chunk_NN/subchunk_MM/`. Chunk dirs
are sorted numerically (`chunk_100` comes after `chunk_99`).

**Output** -- JOB LEVEL ONLY, under
`AnalysisDistributions/multi_file_plots_charge_light_matching/SignalBackground_Distributions_BeforeCosmicTagger_Data_OnBeam_InputRootFiles/combined_apa_<date>_<time>/job_summary/`:

```
reco_cluster_selection_flow_job_Combined.png, selection_flow_counts.txt
selected_reco_histograms.root
selected_clusters.txt   (one line per selected cluster / candidate)
bee_links_<population>.txt, summary.txt
AfterBeamWindowCut/, NuCandidate/, NuCandidate_NuMuScore_gt0/, NuCandidate_NuMuScore_gt0p9/
  Reco_Cluster_Energy/<w>MeV/   selected_reco_energy_*.png (+ info .txt, .root)
  Reco_Neutrino_Energy/<w>MeV/  kine_reco_Enu_*.png (+ _info.txt)   (not AfterBeamWindowCut)
  Length/, NPoints/, Flash_PE/, Flash_Time/, NuMu_Score/, NuE_Score/   plot (+ _info.txt);
                                Flash_Time also the -10..10 us zoom-out
                                (AfterBeamWindowCut, NuCandidate)
bee_set_<population>/event_map.txt, bee_set_<population>.zip   (one level up, one per population)
```

In [ ]:
# Run scope.
#   files  -> how many subchunk dirs to read ("all" or 1/2/3/...)
#   events -> how many ROOT files (= events) per subchunk dir ("all" or 1/2/3/...)
# Every subchunk of every chunk in ROOT_SOURCE_CHUNKS (10 chunks x 100 files = 1000 events).
files  = "all"
events = "all"

# Optional: only this event, as (run, subrun, event). None = no filter.
target_run_subrun_event = None

In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions/, one level below the
# repository root where the pipeline modules are. Resolve both explicitly so
# the notebook runs whether Jupyter was started here or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions":
    NB_DIR = NB_DIR / "AnalysisDistributions"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")

In [ ]:
# Pipeline modules. The selection comes from the tracking-pr ROOT file; the only
# JSON read is clustering-global, for the selected clusters' charge (reco energy).
from readfiles import (read_tracking_pr_root, find_tracking_pr_files, list_subchunk_dirs,
                       read_cluster_global_from_json)
from DrawRecoTrueClusterCount import DrawRecoClusterSelectionFlow
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US
from draw_signal_background import (DEFAULT_RECO_CUTS_LABEL, BIN_WIDTHS_MEV, energy_bin_edges,
                                    RECO_WORK_FUNCTION_EV, RECO_RECOMBINATION_FACTOR)
from draw_selection_performance import (
    plot_directory, PLOT_X_MAX_MEV, draw_selected_data_points,
    write_selected_data_points_info, write_selected_data_points_root,
    draw_selected_reco_energy, write_selected_reco_info, write_selected_reco_root,
    draw_2d_points,
)

In [ ]:
# ========================================================================
# INPUT SAMPLE -- on-beam (beam-on) real data, tracking-pr ROOT files
# ========================================================================
SAMPLE_NAME = "OnBeam_Sample"
SAMPLE_KIND = "Data"   # figure titles: "Data" or "MC"
# Reco-energy plots as black data points. MC draws them as truth-category
# stacks instead (its own cells), so it turns these off.
DRAW_RECO_ENERGY_POINTS = True
# BEE sets (the last cell): every earlier run already uploaded them, so they are
# not rebuilt by default. True rebuilds and uploads them as before.
MAKE_BEE_SETS = False
SAMPLE_DIR  = Path("/Volumes/My Passport/Research_Life/Experiment/SBND/"
                   "Wirecell_Reconstruction/Samples/r3-beam-on-2026-09-10")

# chunk_NN dirs to read, each relative to SAMPLE_DIR. bee/chunk_00..09 hold the
# ROOT files of the first 1000 events (next to their zips); after_1000files/
# and tracking-pr/ hold the rest.
ROOT_SOURCE_CHUNKS = [f"bee/chunk_{i:02d}" for i in range(10)]   # chunk_00..chunk_09, 1000 events
# Every chunk of the sample: bee/ (zips), after_1000files/ (zips), tracking-pr/ (ROOT only).
# ROOT_SOURCE_CHUNKS = [str(d.relative_to(SAMPLE_DIR))
#                       for top in ("bee", "after_1000files", "tracking-pr")
#                       for d in sorted((d for d in (SAMPLE_DIR / top).glob("chunk_[0-9]*") if d.is_dir()),
#                                       key=lambda d: int(d.name.split("_")[1]))]

num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory
PLOTBASEDIR = (NB_DIR / "multi_file_plots_charge_light_matching"
               / "SignalBackground_Distributions_BeforeCosmicTagger_Data_OnBeam_InputRootFiles")
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

# ========================================================================
# RECO SELECTIONS -- reco-level only, no cosmic tagger cut (same as the JSON
# notebook), plus the T_tagger neutrino-candidate stage the ROOT file adds.
# ========================================================================
RECO_SELECTION_NOCUTS    = DEFAULT_RECO_CUTS_LABEL   # 'NoCuts': in_scope && is_main
RECO_SELECTION_BEAM      = 'AfterBeamWindowCut'      # + flash_id >= 0, flash time in window
RECO_SELECTION_CANDIDATE = 'NuCandidate'             # + listed in T_tagger
# + at least one credited candidate with its T_tagger reco vertex (nu_x/y/z) inside
# the fiducial volume (selections.py Fiducial_*). The vertex exists only on T_tagger
# rows, so this stage follows NuCandidate. A (0, 0, 0) vertex (no pattern
# recognition output) counts as outside.
RECO_SELECTION_FV = f'{RECO_SELECTION_CANDIDATE}_FiducialRecoVertex'
# + at least one credited candidate with its vertex in the FV AND numu_score > cut --
# one stage per cut, named NuCandidate_FiducialRecoVertex_NuMuScore_gt0, ..._gt0p9.
NUMU_SCORE_CUTS = (0.0, 0.9)
def numu_stage(cut):
    return f"{RECO_SELECTION_FV}_NuMuScore_gt{cut:g}".replace('.', 'p')
RECO_FLOW_STAGES = [(RECO_SELECTION_NOCUTS,    'No cuts'),
                    (RECO_SELECTION_BEAM,      '+ Beam window'),
                    (RECO_SELECTION_CANDIDATE, '+ Nu candidate'),
                    (RECO_SELECTION_FV,        '+ Reco vertex in FV')] + [
                   (numu_stage(cut), f'+ numu_score > {cut:g}') for cut in NUMU_SCORE_CUTS]

# ========================================================================
# HISTOGRAMS -- fixed edges, so every run of this notebook bins the same way.
#   (key, x label, edges, x tick)
# ========================================================================
BEAM_CLUSTER_VARIABLES = [
    ('length_cm',     'Reco Cluster Length (cm)',       np.arange(0.0, 620.0, 20.0),   100.0),
    ('npoints',       'Reco Cluster Points (npoints)',  np.arange(0.0, 20500.0, 500.0), 2500.0),
    ('flash_pe',      'Matched Flash PE',               np.arange(0.0, 31000.0, 1000.0), 5000.0),
    ('flash_time_us', 'Matched Flash Time (us)',
     np.arange(BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US + 1e-9, 0.1), 0.2),
]
# Zoomed-out flash time: EVERY NoCuts cluster with a flash, -10..10 us, the
# beam window shaded -- shows the selected clusters against the activity just
# outside the window, which the AfterBeamWindowCut plot cannot (it has none).
FLASH_TIME_ZOOM_EDGES = np.arange(-10.0, 10.0 + 1e-9, 0.2)
CANDIDATE_SCORE_VARIABLES = [
    ('numu_score', 'numu_score (BDT)', np.arange(-10.0, 5.5, 0.5), 1.0),
    ('nue_score',  'nue_score (BDT)',  np.arange(-16.0, 10.5, 0.5), 2.0),
]
SELECTION_BIN_WIDTHS_MEV = BIN_WIDTHS_MEV + (50.0,)   # kine_reco_Enu: 200, 100, 50 MeV
KINE_ENU_EDGES = {w: energy_bin_edges(w)[energy_bin_edges(w) <= PLOT_X_MAX_MEV]
                  for w in SELECTION_BIN_WIDTHS_MEV}

# Further per-candidate variables, drawn like kine_reco_Enu at every candidate
# stage, one entry per credited candidate that has a value:
#   (record key, output dir, x label, edges -- None: KINE_ENU_EDGES at every
#    width, each in <dir>/<w>MeV/ -- x tick)
#   n_muons      T_kine particles with |PDG| == 13 (0 when it lists none)
#   muon_energy  kine_energy_particle of the leading (highest-energy) muon; no
#                entry for a candidate without a muon
#   nu_x/y/z     T_tagger's reco neutrino vertex; no entry when it is exactly
#                (0, 0, 0), the default of a candidate with no pattern
#                recognition output (numu_score -1.94)
CANDIDATE_EXTRA_VARIABLES = [
    ('n_muons',     'Reco_Muon_Multiplicity', 'Reco Muon Multiplicity (|PDG| = 13 in T_kine)',
     np.arange(-0.5, 6.0, 1.0), 1.0),
    ('muon_energy', 'Reco_Muon_Energy', 'Leading Reco Muon Energy, kine_energy_particle (MeV)',
     None, 500.0),
    ('nu_x', 'Reco_Vertex_X', 'Reco Neutrino Vertex x (cm)', np.arange(-200.0, 210.0, 10.0), 50.0),
    ('nu_y', 'Reco_Vertex_Y', 'Reco Neutrino Vertex y (cm)', np.arange(-200.0, 210.0, 10.0), 50.0),
    ('nu_z', 'Reco_Vertex_Z', 'Reco Neutrino Vertex z (cm)', np.arange(0.0, 510.0, 10.0), 100.0),
]

# Muon direction: cos(theta) to the beam (+z) of the candidate's muon, from its
# T_rec_charge trajectory points (particle_id 13) within MUON_DIR_LENGTH_CM of the
# reco vertex -- the start of the track. Agrees with Wire-Cell's own
# cosmict_2_angle_beam wherever that is filled. With two muons both starts are
# averaged (~12% of candidates).
MUON_DIR_LENGTH_CM = 30.0
CANDIDATE_EXTRA_VARIABLES.append(
    ('muon_cos_theta', 'Muon_Cosmu', r'Reco Muon $\cos\theta$ to the Beam (z)',
     np.arange(-1.0, 1.0001, 0.1), 0.5))


# Particle, pi0, track-length and sub-BDT variables (T_kine particle list,
# T_rec_charge residual range, T_tagger scores), one entry per candidate with a
# value. Track length = the largest residual range among that particle type's
# points (the longest such track). pi0 mass only where kine_pio_flag == 1.
SUB_SCORE_EDGES = np.arange(-0.7, 0.4001, 0.025)
CANDIDATE_EXTRA_VARIABLES += [
    ('n_protons', 'Reco_Proton_Multiplicity', 'Reco Proton Multiplicity (|PDG| = 2212 in T_kine)',
     np.arange(-0.5, 8.0, 1.0), 1.0),
    ('n_charged_pions', 'Reco_ChargedPion_Multiplicity',
     'Reco Charged-Pion Multiplicity (|PDG| = 211 in T_kine)', np.arange(-0.5, 6.0, 1.0), 1.0),
    ('n_em_objects', 'Reco_EMObject_Multiplicity', 'Reco EM-Object Multiplicity (|PDG| = 11 in T_kine: any EM-like object, mostly few-MeV fragments)',
     np.arange(-0.5, 15.0, 1.0), 2.0),
    ('proton_energy', 'Reco_Proton_Energy', 'Leading Reco Proton Energy, kine_energy_particle (MeV)',
     np.arange(0.0, 1025.0, 25.0), 200.0),
    ('pi0_mass', 'Pi0_Mass', r'Reco $\pi^0$ Mass, kine_pio_mass (MeV)', np.arange(0.0, 410.0, 10.0), 50.0),
    ('muon_length', 'Muon_Length', 'Longest Reco Muon Track Length, max residual range (cm)',
     np.arange(0.0, 510.0, 10.0), 100.0),
    ('proton_length', 'Proton_Length', 'Longest Reco Proton Track Length, max residual range (cm)',
     np.arange(0.0, 102.0, 2.0), 20.0),
    ('numu_1_score', 'NuMu_1_Score', 'numu_1_score (sub-BDT)', SUB_SCORE_EDGES, 0.2),
    ('numu_2_score', 'NuMu_2_Score', 'numu_2_score (sub-BDT)', SUB_SCORE_EDGES, 0.2),
    ('numu_3_score', 'NuMu_3_Score', 'numu_3_score (sub-BDT)', SUB_SCORE_EDGES, 0.2),
    ('cosmict_score', 'Cosmict_Score', 'cosmict_score (cosmic-tagger BDT)',
     np.arange(-1.0, 1.0001, 0.05), 0.5),
]

# dQ/dx: each muon / proton trajectory point's charge q (T_rec_charge; it holds no
# dx, so q is proportional to dQ/dx but not calibrated) against its residual range
# rr: a 2D plot per stage, plus q's 1D distribution (MC: by truth category).
DQDX_PARTICLES = (('mu', 13, 'Muon', 'DQDX_Muon'), ('p', 2212, 'Proton', 'DQDX_Proton'))
DQDX_Q_EDGES = np.arange(0.0, 15250.0, 250.0)
DQDX_RR_EDGES = np.arange(0.0, 102.0, 2.0)

# Which variables this run draws. None: everything (the full job). A tuple of
# variable keys, e.g. ('muon_cos_theta',): ONLY those -- the selection still runs,
# and in MC so does the truth matching (the stacks need the categories), but the
# reco energy from the staged JSONs, the -10..10 us truth matching, every other
# plot and the efficiencies are skipped. Used to time a one-variable job.
PLOT_VARIABLES = None
DRAW_ALL_VARIABLES = PLOT_VARIABLES is None
PLOTTED_EXTRA_VARIABLES = [v for v in CANDIDATE_EXTRA_VARIABLES
                           if DRAW_ALL_VARIABLES or v[0] in PLOT_VARIABLES]
DRAW_DQDX = DRAW_ALL_VARIABLES or 'dqdx' in PLOT_VARIABLES
NEED_REC_CHARGE = DRAW_DQDX or any(v[0] in ('muon_cos_theta', 'muon_length', 'proton_length')
                                   for v in PLOTTED_EXTRA_VARIABLES)

# Output layout under job_summary/<stage>/, one directory per quantity -- as in
# the MC notebook, minus its truth-category levels (data has no truth):
#   Reco_Cluster_Energy/<w>MeV/      reco energy points (+ info .txt, .root)
#   Reco_Neutrino_Energy/<w>MeV/     kine_reco_Enu (candidate stages)
#   Length/, NPoints/, Flash_PE/, Flash_Time/, NuMu_Score/, NuE_Score/
RECO_ENERGY_DIRNAME = 'Reco_Cluster_Energy'
NU_ENERGY_DIRNAME   = 'Reco_Neutrino_Energy'
VARIABLE_DIRNAMES = {'length_cm': 'Length', 'npoints': 'NPoints', 'flash_pe': 'Flash_PE',
                     'flash_time_us': 'Flash_Time', 'numu_score': 'NuMu_Score', 'nue_score': 'NuE_Score'}
VARIABLE_SUBDIR = None   # MC: its signal-definition dir, holding the truth stacks

print("Configuration:")
print(f"  sample:        {SAMPLE_DIR}")
print(f"  source chunks: {ROOT_SOURCE_CHUNKS}")
print(f"  plot base dir: {PLOTBASEDIR}")
print(f"  files (subchunks): {files}   events (ROOT files) per subchunk: {events}")
if target_run_subrun_event is not None:
    print(f"  ONLY event (run, subrun, event) = {target_run_subrun_event}")
print(f"\nSelections: {' -> '.join(s for _, s in RECO_FLOW_STAGES)}")
print(f"  beam window: {BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us; flash_id < 0 is cut")
print(f"  cosmic tagger cut: NOT applied (same as the JSON notebook)")

In [ ]:
# Subchunk dirs, numerically sorted, then their ROOT files.
subchunk_dirs = []
for source_chunk in ROOT_SOURCE_CHUNKS:
    subchunk_dirs += list_subchunk_dirs(SAMPLE_DIR / source_chunk)
if num_files_to_process is not None:
    subchunk_dirs = subchunk_dirs[:num_files_to_process]

input_units = find_tracking_pr_files(subchunk_dirs)
if num_events_to_process is not None:
    input_units = [(d, fs[:num_events_to_process]) for d, fs in input_units]

# The staged zip JSONs: staging/<chunk>__<subchunk>/data/<k>/, k from that
# dir's event_map.txt. Used for the reco-energy charge sums and for BEE.
STAGING_ROOT = SAMPLE_DIR / "staging"
_staged_maps = {}


def staged_key(unit, run, subrun, event):
    """(staged dir name, k) for an event, or None if its JSONs are not staged."""
    parts = Path(unit).parts                      # ('bee', 'chunk_00', 'subchunk_03')
    if len(parts) != 3 or parts[0] != 'bee':
        return None
    staged = f"{parts[1]}__{parts[2]}"
    if staged not in _staged_maps:
        emap = STAGING_ROOT / staged / "event_map.txt"
        _staged_maps[staged] = ({tuple(int(v) for v in row[1:4]): int(row[0])
                                 for row in (l.split() for l in emap.read_text().splitlines())
                                 if row and not row[0].startswith('#')}
                                if emap.exists() else {})
    k = _staged_maps[staged].get((run, subrun, event))
    return None if k is None else (staged, k)


n_root_files = sum(len(fs) for _, fs in input_units)
print(f"{len(input_units)} subchunk dir(s), {n_root_files} ROOT file(s)")
for sub_dir, root_files in input_units:
    print(f"  {sub_dir.relative_to(SAMPLE_DIR)}: {len(root_files)} file(s)")

In [ ]:
# ============================================================================
# MAIN LOOP -- one ROOT file = one event
# ============================================================================
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}\n")

job_flow_counts      = {label: 0 for label, _ in RECO_FLOW_STAGES}
job_events           = []   # (unit, (run, subrun, event)) of every event read
job_beam_records     = []   # one dict per AfterBeamWindowCut cluster
job_nocut_flash_times = []  # flash time of every NoCuts cluster that has a flash
job_candidate_records = []  # T_tagger candidates credited to a beam-window main cluster
job_uncredited_candidates = []  # the rest -- summary only, not in the flow or plots
events_with_candidate   = 0
events_without_json     = []   # beam-window events whose clustering-global is not staged
total_events_processed  = 0
unreadable_files        = []


from selections import (in_fiducial_volume, FIDUCIAL_CATHODE_EXCLUSION_CM,
                        Fiducial_X_MIN, Fiducial_X_MAX, Fiducial_Y_MIN, Fiducial_Y_MAX,
                        Fiducial_Z_MIN, Fiducial_Z_MAX)


def reco_vertex_in_fv(record):
    """The candidate's T_tagger reco vertex is inside the fiducial volume -- the same
    TPC-aware test as the true vertex (selections.in_fiducial_volume), so |x| <=
    FIDUCIAL_CATHODE_EXCLUSION_CM (cathode + 2 cm) is out. NaN -- no vertex, or the
    (0, 0, 0) default -- fails."""
    return bool(in_fiducial_volume(record['nu_x'], record['nu_y'], record['nu_z']))


def candidate_extras(tagger, j, kine, k, rec_charge=None):
    """Muon multiplicity / leading muon energy (T_kine row k), the reco vertex
    (T_tagger row j) and the muon cos(theta) to the beam (T_rec_charge) of one
    candidate -- see CANDIDATE_EXTRA_VARIABLES."""
    out = {'n_muons': np.nan, 'muon_energy': np.nan, 'nu_x': np.nan, 'nu_y': np.nan, 'nu_z': np.nan,
           'muon_cos_theta': np.nan, 'n_protons': np.nan, 'n_charged_pions': np.nan,
           'n_em_objects': np.nan, 'proton_energy': np.nan, 'pi0_mass': np.nan,
           'muon_length': np.nan, 'proton_length': np.nan,
           'mu_q': np.zeros(0), 'mu_rr': np.zeros(0), 'p_q': np.zeros(0), 'p_rr': np.zeros(0)}
    for score in ('numu_1_score', 'numu_2_score', 'numu_3_score', 'cosmict_score'):
        out[score] = float(tagger[score][j]) if score in tagger else np.nan
    if rec_charge and 'rr' in rec_charge:
        own = rec_charge['cluster_id'] == int(tagger['cluster_id'][j])
        pid = np.abs(rec_charge['particle_id'])
        for short, pdg, _label, _dir in DQDX_PARTICLES:
            points = own & (pid == pdg)
            out[f'{short}_q'] = rec_charge['q'][points].astype(float)
            out[f'{short}_rr'] = rec_charge['rr'][points].astype(float)
        if len(out['mu_rr']):
            out['muon_length'] = float(out['mu_rr'].max())
        if len(out['p_rr']):
            out['proton_length'] = float(out['p_rr'].max())
    if k is not None and 'kine_particle_type' in kine:
        types = np.abs(np.asarray(kine['kine_particle_type'][k]))
        energies = np.asarray(kine['kine_energy_particle'][k], dtype=float)
        is_muon = types == 13
        out['n_muons'] = int(is_muon.sum())
        out['n_protons'] = int((types == 2212).sum())
        out['n_charged_pions'] = int((types == 211).sum())
        out['n_em_objects'] = int((types == 11).sum())
        if (types == 2212).any():
            out['proton_energy'] = float(energies[types == 2212].max())
        if ('kine_pio_flag' in kine and int(kine['kine_pio_flag'][k]) == 1
                and float(kine['kine_pio_mass'][k]) > 0):
            out['pi0_mass'] = float(kine['kine_pio_mass'][k])
        if is_muon.any():
            out['muon_energy'] = float(energies[is_muon].max())
    if 'nu_x' in tagger:
        vertex = [float(tagger[c][j]) for c in ('nu_x', 'nu_y', 'nu_z')]
        if any(vertex):
            out['nu_x'], out['nu_y'], out['nu_z'] = vertex
    if rec_charge and np.isfinite(out['nu_x']):
        muon = ((rec_charge['cluster_id'] == int(tagger['cluster_id'][j]))
                & (np.abs(rec_charge['particle_id']) == 13))
        if muon.sum() >= 2:
            points = np.column_stack([rec_charge[c][muon] for c in ('x', 'y', 'z')]).astype(float)
            vertex = np.array([out['nu_x'], out['nu_y'], out['nu_z']])
            distance = np.linalg.norm(points - vertex, axis=1)
            start = points[distance < MUON_DIR_LENGTH_CM]
            if len(start) < 2:
                start = points[np.argsort(distance)[:5]]
            direction = start.mean(axis=0) - vertex
            norm = np.linalg.norm(direction)
            if norm > 0:
                out['muon_cos_theta'] = float(direction[2] / norm)
    return out

for unit_idx, (sub_dir, root_files) in enumerate(input_units):
    unit_name = str(sub_dir.relative_to(SAMPLE_DIR))
    for root_file in root_files:
        event_data = read_tracking_pr_root(root_file, rec_charge=NEED_REC_CHARGE)
        if event_data is None:
            unreadable_files.append(root_file)
            continue
        rse = (event_data['run'], event_data['subrun'], event_data['event'])
        if target_run_subrun_event is not None and rse != tuple(target_run_subrun_event):
            continue
        job_events.append((unit_name, rse))

        clusters = event_data['cluster']
        n_rows = len(clusters.get('cluster_id', []))

        # NoCuts: the main clusters in scope -- the JSON notebook's clusters.
        nocut = ((clusters['in_scope'] == 1) & (clusters['is_main'] == 1)) if n_rows else np.zeros(0, bool)
        # + beam window: needs a matched flash, inside the window.
        beam = (nocut & (clusters['flash_id'] >= 0)
                & (clusters['flash_time_us'] >= BEAM_WINDOW_MIN_US)
                & (clusters['flash_time_us'] <= BEAM_WINDOW_MAX_US)) if n_rows else nocut
        beam_ids = set(clusters['cluster_id'][beam].tolist()) if n_rows else set()
        if n_rows:
            job_nocut_flash_times.extend(
                clusters['flash_time_us'][nocut & (clusters['flash_id'] >= 0)].tolist())

        # Reco energy needs the cluster's summed charge, which the ROOT file does
        # not carry: take it from the staged clustering-global JSON, whose
        # cluster_id is the same id as T_cluster's (checked on off-beam chunk_00).
        cluster_charge = {}
        if beam_ids and DRAW_ALL_VARIABLES and DRAW_RECO_ENERGY_POINTS:   # reco energy only
            key = staged_key(unit_name, *rse)
            json_path = (STAGING_ROOT / key[0] / "data" / str(key[1])
                         / f"{key[1]}-clustering-global.json") if key else None
            if json_path is not None and json_path.exists():
                _x, _y, _z, json_ids, json_q, _real = read_cluster_global_from_json(json_path)
                cluster_charge = {cid: float(json_q[json_ids == cid].sum()) for cid in beam_ids}
            else:
                events_without_json.append(rse)

        for i in np.flatnonzero(beam):
            job_beam_records.append({
                'unit': unit_name, 'run': rse[0], 'subrun': rse[1], 'event': rse[2],
                'cluster_id': int(clusters['cluster_id'][i]),
                'total_charge': cluster_charge.get(int(clusters['cluster_id'][i]), np.nan),
                **{key: float(clusters[key][i]) for key, *_ in BEAM_CLUSTER_VARIABLES},
            })

        # + Nu candidate: T_tagger rows, joined to T_kine on nu_index. A
        # candidate is credited to a beam-window main cluster when it IS one
        # ('main'), or when it is an associated cluster sharing its flash with
        # one ('associated' -- in the JSON that main cluster absorbs it). Any
        # other candidate (in practice: out-of-scope main clusters, tiny and
        # with empty kinematics) is not in the NoCuts population, so it is
        # listed in the summary but kept out of the flow and the plots.
        tagger, kine = event_data['tagger'], event_data['kine']
        kine_by_nu = ({int(n): j for j, n in enumerate(kine['nu_index'])}
                      if kine and 'nu_index' in kine else {})
        row_of = {int(c): i for i, c in enumerate(clusters['cluster_id'])} if n_rows else {}
        beam_main_by_flash = {int(clusters['flash_id'][i]): int(clusters['cluster_id'][i])
                              for i in np.flatnonzero(beam)}
        credited_ids = set()
        fv_ids = set()                                      # credited clusters with a vertex in the FV
        numu_ids = {cut: set() for cut in NUMU_SCORE_CUTS}   # ... and numu_score > cut
        for j in range(len(tagger.get('cluster_id', []))):
            cid = int(tagger['cluster_id'][j])
            i = row_of.get(cid)
            if cid in beam_ids:
                match, credited = 'main', cid
            elif (i is not None and clusters['is_associated'][i] == 1
                  and int(clusters['flash_id'][i]) in beam_main_by_flash):
                match, credited = 'associated', beam_main_by_flash[int(clusters['flash_id'][i])]
            else:
                match, credited = ('out_of_scope' if i is not None and clusters['in_scope'][i] == 0
                                   else 'not_in_beam_window'), None
            k = kine_by_nu.get(int(tagger['nu_index'][j])) if 'nu_index' in tagger else None
            record = {
                'unit': unit_name, 'run': rse[0], 'subrun': rse[1], 'event': rse[2],
                'cluster_id': cid, 'match': match,
                'credited_cluster_id': credited if credited is not None else -1,
                'numu_score': float(tagger['numu_score'][j]),
                'nue_score':  float(tagger['nue_score'][j]),
                'kine_reco_Enu': float(kine['kine_reco_Enu'][k]) if k is not None else np.nan,
                **candidate_extras(tagger, j, kine, k, event_data.get('rec_charge')),
                'npoints': int(clusters['npoints'][i]) if i is not None else -1,
                'flash_time_us': float(clusters['flash_time_us'][i]) if i is not None else np.nan,
                'flash_id': int(clusters['flash_id'][i]) if i is not None else -1,
                # charge of the beam-window cluster the candidate is credited to
                'total_charge': cluster_charge.get(credited, np.nan) if credited is not None else np.nan,
            }
            record['in_fv'] = reco_vertex_in_fv(record)
            if credited is None:
                job_uncredited_candidates.append(record)
            else:
                credited_ids.add(credited)
                if record['in_fv']:
                    fv_ids.add(credited)
                    for cut in NUMU_SCORE_CUTS:
                        if record['numu_score'] > cut:
                            numu_ids[cut].add(credited)
                job_candidate_records.append(record)
        events_with_candidate += bool(credited_ids)

        job_flow_counts[RECO_SELECTION_NOCUTS]    += int(nocut.sum())
        job_flow_counts[RECO_SELECTION_BEAM]      += int(beam.sum())
        job_flow_counts[RECO_SELECTION_CANDIDATE] += len(credited_ids)
        job_flow_counts[RECO_SELECTION_FV]        += len(fv_ids)
        for cut in NUMU_SCORE_CUTS:
            job_flow_counts[numu_stage(cut)] += len(numu_ids[cut])
        total_events_processed += 1

        print(f"  r{rse[0]} s{rse[1]} e{rse[2]}: reco clusters={int(nocut.sum())} "
              f"(beam window={int(beam.sum())}, nu candidates={len(credited_ids)}"
              + (f"; T_tagger rows={len(tagger['cluster_id'])}" if tagger else "") + ")")

    print(f"  ==> unit {unit_idx + 1}/{len(input_units)} done: {unit_name}  "
          f"({total_events_processed} events, {len(job_beam_records)} beam-window clusters, "
          f"{(time.time() - job_start_time) / 60:.1f} min elapsed)")

print(f"\nJOB: {total_events_processed} event(s), "
      + ", ".join(f"{s}={job_flow_counts[l]}" for l, s in RECO_FLOW_STAGES))
from collections import Counter
match_counts = Counter(r['match'] for r in job_candidate_records + job_uncredited_candidates)
# Each stage's populations, in one place:
#   STAGE_CANDIDATES     the credited T_tagger candidates, at NuCandidate and the
#                        numu_score stages (a cluster passes a numu stage when at
#                        least one candidate credited to it does)
#   STAGE_CLUSTERS       each stage's beam-window clusters, one entry per cluster
#   STAGE_SCORE_RECORDS  the score plots' entries: at AfterBeamWindowCut every
#                        T_tagger row whose own flash is in the beam window,
#                        credited or not (a cluster without a row has no score);
#                        at the later stages, that stage's candidates
STAGE_CANDIDATES = {RECO_SELECTION_CANDIDATE: job_candidate_records,
                    RECO_SELECTION_FV: [r for r in job_candidate_records if r['in_fv']]}
for cut in NUMU_SCORE_CUTS:
    STAGE_CANDIDATES[numu_stage(cut)] = [r for r in job_candidate_records
                                         if r['in_fv'] and r['numu_score'] > cut]

def credited_clusters(candidates):
    keys = {(r['unit'], r['run'], r['subrun'], r['event'], r['credited_cluster_id']) for r in candidates}
    return [r for r in job_beam_records
            if (r['unit'], r['run'], r['subrun'], r['event'], r['cluster_id']) in keys]

STAGE_CLUSTERS = {RECO_SELECTION_BEAM: job_beam_records,
                  **{stage: credited_clusters(c) for stage, c in STAGE_CANDIDATES.items()}}
for stage, clusters in STAGE_CLUSTERS.items():   # the flow counted the same clusters
    assert len(clusters) == job_flow_counts[stage], (stage, len(clusters), job_flow_counts[stage])
tagger_beam_records = [r for r in job_candidate_records + job_uncredited_candidates
                       if BEAM_WINDOW_MIN_US <= r['flash_time_us'] <= BEAM_WINDOW_MAX_US]
STAGE_SCORE_RECORDS = {RECO_SELECTION_BEAM: tagger_beam_records, **STAGE_CANDIDATES}
print(f"T_tagger candidates: {len(job_candidate_records) + len(job_uncredited_candidates)} "
      f"({', '.join(f'{m}={n}' for m, n in sorted(match_counts.items()))}); "
      f"{len(job_uncredited_candidates)} not credited to a beam-window cluster "
      f"-- listed in summary.txt, not in the flow or plots")
if unreadable_files:
    print(f"WARNING: {len(unreadable_files)} unreadable ROOT file(s): {unreadable_files[:5]}")

In [ ]:
# ============================================================================
# JOB-LEVEL PLOTS
# ============================================================================
job_output_dir = output_dir / "job_summary"
# Every stage gets its own directory directly under job_summary/; the
# stage-independent files (flow, histogram ROOT files, cluster list, BEE
# links) sit beside them.
reco_dir = job_output_dir
reco_dir.mkdir(parents=True, exist_ok=True)
root_histograms = {}


def variable_dir(stage, dirname, bin_width=None):
    """job_summary/<stage>/<dirname>[/<w>MeV][/<VARIABLE_SUBDIR>], created --
    the scores and kine_reco_Enu."""
    return plot_directory(reco_dir / stage, dirname, bin_width, None, VARIABLE_SUBDIR)


def cluster_variable_dir(stage, key):
    """job_summary/<stage>/<Variable>/, created -- the cluster variables, whose
    MC truth stacks sit one level down, one dir per signal definition."""
    return plot_directory(reco_dir / stage, VARIABLE_DIRNAMES[key])

# 1) Selection flow, one Total bar per stage.
DrawRecoClusterSelectionFlow(
    [{'key': label, 'stage': stage, 'geometry': False, 'total': job_flow_counts[label]}
     for label, stage in RECO_FLOW_STAGES],
    reco_dir, "Job Level", "job", "Combined", include_geometry_cuts=False)
with open(reco_dir / "selection_flow_counts.txt", "w") as f:
    n0 = job_flow_counts[RECO_SELECTION_NOCUTS]
    f.write(f"RECO CLUSTER SELECTION FLOW (Job Level) -- {total_events_processed} events\n")
    for label, stage in RECO_FLOW_STAGES:
        n = job_flow_counts[label]
        f.write(f"  {stage:<18s} {label:<20s} {n:8d}   "
                f"{(100.0 * n / n0 if n0 else 0.0):6.2f}%\n")
    f.write(f"\n'+ Nu candidate' counts beam-window main clusters credited with >= 1 T_tagger\n"
            f"candidate (the candidate itself, or an associated cluster sharing its flash).\n")
    f.write(f"events with >= 1 credited candidate: {events_with_candidate}\n")
    f.write(f"T_tagger candidates by match: "
            f"{', '.join(f'{m}={n}' for m, n in sorted(match_counts.items()))}\n")
    f.write(f"not credited (not in the flow or plots): {len(job_uncredited_candidates)}\n")

if DRAW_ALL_VARIABLES:
    # 2) Cluster variables -- length, npoints, flash PE, flash time -- at every
    #    stage, one entry per selected cluster.
    for stage, clusters in STAGE_CLUSTERS.items():
        for key, xlabel, edges, tick in BEAM_CLUSTER_VARIABLES:
            out_dir = cluster_variable_dir(stage, key)
            values = [r[key] for r in clusters]
            name = f"{key}_{stage}_job_Combined"
            draw_selected_data_points(values, edges, out_dir, name, xlabel,
                                      f'Selected Reco Clusters ({SAMPLE_KIND}, ROOT) -- reco: {stage}',
                                      x_tick=tick)
            write_selected_data_points_info(values, edges, out_dir / f"{name}_info.txt",
                                            f"{key} -- {stage} reco clusters (Job Level)")
            root_histograms[f"{stage}/{key}"] = (values, edges)

    # flash_time_us, zoomed out to -10..10 us, the beam window shaded:
    #   AfterBeamWindowCut -- every NoCuts cluster with a flash
    #   NuCandidate        -- every T_tagger row whose cluster has a flash, credited
    #                         or not (no beam-window cut in this view)
    flash_time_dir = cluster_variable_dir(RECO_SELECTION_BEAM, 'flash_time_us')
    name = f"flash_time_us_{RECO_SELECTION_BEAM}_zoomout_-10to10us_job_Combined"
    in_zoom = [t for t in job_nocut_flash_times if -10.0 <= t <= 10.0]   # legend counts what is shown
    draw_selected_data_points(in_zoom, FLASH_TIME_ZOOM_EDGES, flash_time_dir, name,
                              'Matched Flash Time (us)',
                              f'Reco Clusters with a Flash ({SAMPLE_KIND}, ROOT) -- {RECO_SELECTION_NOCUTS}, -10 to 10 us',
                              label=f'{RECO_SELECTION_NOCUTS} clusters, -10 to 10 us', x_tick=2.0,
                              bands=[(BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US,
                                      f'Beam window {BEAM_WINDOW_MIN_US}-{BEAM_WINDOW_MAX_US} us '
                                      f'({job_flow_counts[RECO_SELECTION_BEAM]} selected)')])
    write_selected_data_points_info(in_zoom, FLASH_TIME_ZOOM_EDGES, flash_time_dir / f"{name}_info.txt",
                                    f"flash_time_us -- {RECO_SELECTION_NOCUTS} clusters with a flash, "
                                    f"-10..10 us (Job Level); beam window "
                                    f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us", unit='us')
    root_histograms[f"{RECO_SELECTION_NOCUTS}/flash_time_us_zoomout"] = (in_zoom, FLASH_TIME_ZOOM_EDGES)

    flash_time_dir = cluster_variable_dir(RECO_SELECTION_CANDIDATE, 'flash_time_us')
    name = f"flash_time_us_{RECO_SELECTION_CANDIDATE}_zoomout_-10to10us_job_Combined"
    tagger_flash_times = [r['flash_time_us'] for r in job_candidate_records + job_uncredited_candidates
                          if r['flash_id'] >= 0]
    in_zoom = [t for t in tagger_flash_times if -10.0 <= t <= 10.0]
    n_in_window = sum(BEAM_WINDOW_MIN_US <= t <= BEAM_WINDOW_MAX_US for t in in_zoom)
    draw_selected_data_points(in_zoom, FLASH_TIME_ZOOM_EDGES, flash_time_dir, name,
                              'Matched Flash Time (us)',
                              f'T_tagger Rows with a Flash ({SAMPLE_KIND}, ROOT) -- -10 to 10 us',
                              ylabel='Number of T_tagger Rows',
                              label='T_tagger rows, -10 to 10 us', x_tick=2.0,
                              bands=[(BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US,
                                      f'Beam window {BEAM_WINDOW_MIN_US}-{BEAM_WINDOW_MAX_US} us '
                                      f'({n_in_window} rows)')])
    write_selected_data_points_info(in_zoom, FLASH_TIME_ZOOM_EDGES, flash_time_dir / f"{name}_info.txt",
                                    f"flash_time_us -- every T_tagger row whose cluster has a flash, "
                                    f"-10..10 us (Job Level); beam window "
                                    f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us", unit='us')
    root_histograms[f"{RECO_SELECTION_CANDIDATE}/flash_time_us_zoomout"] = (in_zoom, FLASH_TIME_ZOOM_EDGES)

if DRAW_ALL_VARIABLES:
    # 3) kine_reco_Enu at every candidate stage, only where filled (> 0).
    n_enu_unfilled_by_stage = {}
    for stage, candidates in STAGE_CANDIDATES.items():
        enu_all = np.array([r['kine_reco_Enu'] for r in candidates], dtype=float)
        enu = enu_all[enu_all > 0]
        n_enu_unfilled_by_stage[stage] = int(np.sum(~(enu_all > 0)))
        if n_enu_unfilled_by_stage[stage]:
            print(f"  {stage}: {n_enu_unfilled_by_stage[stage]} of {len(enu_all)} candidate(s) have "
                  f"kine_reco_Enu <= 0 (kinematics not filled) -- left out of the Enu plots")
        for bin_width in SELECTION_BIN_WIDTHS_MEV:
            out_dir = variable_dir(stage, NU_ENERGY_DIRNAME, bin_width)
            edges = KINE_ENU_EDGES[bin_width]
            name = f"kine_reco_Enu_{stage}_{bin_width:.0f}MeV_job_Combined"
            draw_selected_data_points(enu, edges, out_dir, name, 'Reconstructed Neutrino Energy, kine_reco_Enu (MeV)',
                                      f'Neutrino Candidates ({SAMPLE_KIND}, ROOT) -- reco: {stage}',
                                      ylabel='Number of Candidates', label='Candidates, Enu filled',
                                      x_tick=500.0)
            write_selected_data_points_info(
                enu, edges, out_dir / f"{name}_info.txt",
                f"kine_reco_Enu -- {stage} (Job Level); "
                f"{n_enu_unfilled_by_stage[stage]} candidate(s) with kine_reco_Enu <= 0 not included", unit='MeV')
            root_histograms[f"{stage}/kine_reco_Enu_{bin_width:.0f}MeV"] = (enu, edges)
n_enu_unfilled = sum(1 for r in job_candidate_records if not r['kine_reco_Enu'] > 0)   # summary.txt

# 3b) Muon multiplicity, leading muon energy and reco vertex, at every candidate
#     stage (CANDIDATE_EXTRA_VARIABLES), like kine_reco_Enu.
def extra_variable_bins(edges):
    """[(bin width or None, edges)]: one binning, or KINE_ENU_EDGES at every width."""
    return ([(w, KINE_ENU_EDGES[w]) for w in SELECTION_BIN_WIDTHS_MEV] if edges is None
            else [(None, edges)])


for stage, candidates in STAGE_CANDIDATES.items():
    for key, dirname, xlabel, edges, tick in PLOTTED_EXTRA_VARIABLES:
        values = np.array([r[key] for r in candidates], dtype=float)
        values = values[np.isfinite(values)]
        for bin_width, var_edges in extra_variable_bins(edges):
            out_dir = variable_dir(stage, dirname, bin_width)
            hist_key = key if bin_width is None else f"{key}_{bin_width:.0f}MeV"
            name = f"{key}_{stage}" + (f"_{bin_width:.0f}MeV" if bin_width else "") + "_job_Combined"
            draw_selected_data_points(values, var_edges, out_dir, name, xlabel,
                                      f'Neutrino Candidates ({SAMPLE_KIND}, ROOT) -- reco: {stage}',
                                      ylabel='Number of Candidates', label='Candidates with a value',
                                      x_tick=tick)
            write_selected_data_points_info(
                values, var_edges, out_dir / f"{name}_info.txt",
                f"{key} -- {stage} (Job Level); {len(candidates) - len(values)} of {len(candidates)} "
                f"candidate(s) without a value not included")
            root_histograms[f"{stage}/{hist_key}"] = (values, var_edges)

if DRAW_ALL_VARIABLES:
    # BDT scores, drawn WITHOUT error bars, at every stage (STAGE_SCORE_RECORDS).
    for stage, records in STAGE_SCORE_RECORDS.items():
        label = ('T_tagger rows, flash in beam window' if stage == RECO_SELECTION_BEAM
                 else 'Candidates')
        for key, xlabel, edges, tick in CANDIDATE_SCORE_VARIABLES:
            out_dir = variable_dir(stage, VARIABLE_DIRNAMES[key])
            values = [r[key] for r in records]
            name = f"{key}_{stage}_job_Combined"
            draw_selected_data_points(values, edges, out_dir, name, xlabel,
                                      f'T_tagger Scores ({SAMPLE_KIND}, ROOT) -- reco: {stage}',
                                      ylabel='Number of Candidates', label=label, x_tick=tick,
                                      show_errors=False)
            write_selected_data_points_info(values, edges, out_dir / f"{name}_info.txt",
                                            f"{key} -- {stage}: {label} (Job Level)")
            root_histograms[f"{stage}/{key}"] = (values, edges)

# Reco cluster energy (W * charge / R), the JSON notebook's plot, at every
# stage -- each selected cluster once, even when two T_tagger rows are credited
# to it (the flow's count).
reco_energy_counts = {}
if DRAW_RECO_ENERGY_POINTS and DRAW_ALL_VARIABLES:
    for stage, records in STAGE_CLUSTERS.items():
        with_charge = [r for r in records if np.isfinite(r['total_charge'])]
        reco_energy_counts[stage] = (len(with_charge), len(records) - len(with_charge))
        for bin_width in SELECTION_BIN_WIDTHS_MEV:
            width_dir = plot_directory(reco_dir / stage, RECO_ENERGY_DIRNAME, bin_width)
            energies = draw_selected_reco_energy(with_charge, width_dir, "Job Level", "job", "Combined",
                                                 bin_width=bin_width, reco_cuts_label=stage,
                                                 title=f'Selected Reco Clusters ({SAMPLE_KIND}) -- reco: {stage}')
            write_selected_reco_info(energies, width_dir, "Job Level",
                                     bin_width=bin_width, reco_cuts_label=stage)
            write_selected_reco_root(energies, width_dir, bin_width=bin_width, reco_cuts_label=stage)
        print(f"  reco energy, {stage}: {len(with_charge)} cluster(s)"
              + (f"; {len(records) - len(with_charge)} without a staged clustering-global, not drawn"
                 if len(records) != len(with_charge) else ""))

# 3c) dQ/dx: muon and proton trajectory points of every candidate at each
#     candidate stage -- q against residual range (2D) and q alone (1D).
if DRAW_DQDX:
    for stage, candidates in STAGE_CANDIDATES.items():
        for short, pdg, label, dirname in DQDX_PARTICLES:
            q = np.concatenate([r[f'{short}_q'] for r in candidates] or [np.zeros(0)])
            rr = np.concatenate([r[f'{short}_rr'] for r in candidates] or [np.zeros(0)])
            out_dir = variable_dir(stage, dirname)
            q_label = f'{label} Point Charge q (proportional to dQ/dx)'
            draw_2d_points(rr, q, DQDX_RR_EDGES, DQDX_Q_EDGES, out_dir,
                           f"{short}_q_vs_rr_{stage}_job_Combined", 'Residual Range (cm)', q_label,
                           f'{label} Trajectory Points ({SAMPLE_KIND}, ROOT) -- reco: {stage}')
            name = f"{short}_q_{stage}_job_Combined"
            draw_selected_data_points(q, DQDX_Q_EDGES, out_dir, name, q_label,
                                      f'{label} Trajectory Points ({SAMPLE_KIND}, ROOT) -- reco: {stage}',
                                      ylabel='Number of Points', label=f'{label} points', x_tick=2500.0)
            write_selected_data_points_info(q, DQDX_Q_EDGES, out_dir / f"{name}_info.txt",
                                            f"{short}_q -- {stage}: {label.lower()} trajectory points (Job Level)")
            root_histograms[f"{stage}/{short}_q"] = (q, DQDX_Q_EDGES)
            root_histograms[f"{stage}/{short}_q_vs_rr"] = (rr, q, DQDX_RR_EDGES, DQDX_Q_EDGES)

root_path = write_selected_data_points_root(root_histograms, reco_dir / "selected_reco_histograms.root")

# 4) Every selected cluster / candidate, one line each, to look events up later.
with open(reco_dir / "selected_clusters.txt", "w") as f:
    f.write(f"# {RECO_SELECTION_BEAM} clusters\n# unit\trun\tsubrun\tevent\tcluster_id\t"
            + "\t".join(k for k, *_ in BEAM_CLUSTER_VARIABLES) + "\n")
    for r in job_beam_records:
        f.write("\t".join(str(r[k]) for k in ('unit', 'run', 'subrun', 'event', 'cluster_id'))
                + "\t" + "\t".join(f"{r[k]:.4g}" for k, *_ in BEAM_CLUSTER_VARIABLES) + "\n")
    for title, records in ((f"{RECO_SELECTION_CANDIDATE} (T_tagger, credited -- in the flow and plots)",
                            job_candidate_records),
                           ("T_tagger candidates NOT credited (summary only)", job_uncredited_candidates)):
        f.write(f"\n# {title}\n# unit\trun\tsubrun\tevent\tcluster_id\tmatch\tcredited_cluster_id\t"
                "npoints\tflash_time_us\tkine_reco_Enu\tnumu_score\tnue_score\n")
        for r in records:
            f.write("\t".join(str(r[k]) for k in ('unit', 'run', 'subrun', 'event', 'cluster_id',
                                                   'match', 'credited_cluster_id', 'npoints'))
                    + f"\t{r['flash_time_us']:.4g}\t{r['kine_reco_Enu']:.4g}"
                    f"\t{r['numu_score']:.4g}\t{r['nue_score']:.4g}\n")

print(f"Plots written under {reco_dir}")
print(f"ROOT histograms: {root_path}")

In [ ]:
# ============================================================================
# JOB SUMMARY TEXT FILE
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

summary_lines = ["=" * 80,
                 "JOB SUMMARY -- SELECTED RECO CLUSTERS, DATA, INPUT: tracking-pr ROOT FILES",
                 "=" * 80,
                 f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}",
                 "",
                 "Configuration:",
                 f"  sample:            {SAMPLE_DIR}",
                 f"  source chunks:     {ROOT_SOURCE_CHUNKS}",
                 f"  files (subchunks): {files}   events per subchunk: {events}",
                 f"  target event:      {target_run_subrun_event}",
                 "",
                 "Selections (reco-level only; no cosmic tagger cut):",
                 f"  {RECO_SELECTION_NOCUTS}: T_cluster rows with in_scope == 1 && is_main == 1",
                 f"  {RECO_SELECTION_BEAM}: + flash_id >= 0 && flash_time_us in "
                 f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us",
                 f"  {RECO_SELECTION_CANDIDATE}: + credited with a T_tagger candidate (the cluster "
                 f"itself, or an associated cluster sharing its flash)",
                 f"  {RECO_SELECTION_FV}: + at least one credited candidate with its reco vertex in "
                 f"x [{Fiducial_X_MIN}, -{FIDUCIAL_CATHODE_EXCLUSION_CM}] or "
                 f"[{FIDUCIAL_CATHODE_EXCLUSION_CM}, {Fiducial_X_MAX}] (cathode excluded), "
                 f"y [{Fiducial_Y_MIN}, {Fiducial_Y_MAX}], z [{Fiducial_Z_MIN}, {Fiducial_Z_MAX}] cm",
                 *[f"  {numu_stage(cut)}: + at least one credited candidate with its reco vertex "
                   f"in the FV and numu_score > {cut:g}"
                   for cut in NUMU_SCORE_CUTS],
                 "",
                 "=" * 80,
                 "JOB-LEVEL AGGREGATION",
                 "=" * 80,
                 f"Subchunk dirs read: {len(input_units)}",
                 f"Events processed:   {total_events_processed}",
                 f"Unreadable files:   {len(unreadable_files)}"]
for label, stage in RECO_FLOW_STAGES:
    summary_lines.append(f"  {stage:<18s} {job_flow_counts[label]:8d}")
summary_lines += [f"Reco energy = {RECO_WORK_FUNCTION_EV} eV * charge / {RECO_RECOMBINATION_FACTOR}, "
                  f"charge summed from the staged clustering-global JSON:"]
summary_lines += [f"  {stage}: {n} cluster(s) drawn, {n_missing} without a staged JSON"
                  for stage, (n, n_missing) in reco_energy_counts.items()]
summary_lines += [f"Events with >= 1 credited candidate: {events_with_candidate}",
                  f"T_tagger candidates by match: "
                  f"{', '.join(f'{m}={n}' for m, n in sorted(match_counts.items()))}",
                  f"Credited candidates with kine_reco_Enu <= 0: {n_enu_unfilled} of {len(job_candidate_records)}",
                  "",
                  f"T_tagger candidates NOT credited ({len(job_uncredited_candidates)}) -- not in the "
                  f"NoCuts population, so kept out of the flow and plots:"]
summary_lines += [f"  r{r['run']} s{r['subrun']} e{r['event']}  cluster {r['cluster_id']:>3d}  "
                  f"{r['match']:<18s} npoints={r['npoints']:<5d} flash_time={r['flash_time_us']:.2f} us  "
                  f"kine_reco_Enu={r['kine_reco_Enu']:.1f}"
                  for r in job_uncredited_candidates]
summary_lines += ["",
                  f"ROOT histograms: {root_path}",
                  "",
                  "=" * 80,
                  "JOB RUNTIME",
                  "=" * 80,
                  f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}",
                  f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}",
                  f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)",
                  "=" * 80]

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")

In [ ]:
# ============================================================================
# BEE SETS -- one upload per population, inside this run's directory:
#   bee_set_AfterBeamWindowCut  every event with >= 1 beam-window cluster
#   bee_set_NuCandidate         every event with >= 1 credited candidate
#   bee_set_AfterBeamWindowCut_NotNuCandidate
#                               every event with a beam-window cluster that is
#                               NOT credited with a candidate
# BEE needs the event's JSONs, so each event is taken from the staged zip
# contents in staging/<chunk>__<subchunk>/data/<k>/ (found via that dir's
# event_map.txt). An event with no staged JSONs -- e.g. a ROOT-only file under
# tracking-pr/ -- cannot be shown in BEE and is listed as skipped.
# ============================================================================
from collections import OrderedDict
from build_bee_set_from_links import build_and_upload_set

# Beam-window clusters NOT credited with a candidate -- to check by eye why
# WireCell did not promote them.
credited = {(r['run'], r['subrun'], r['event'], r['credited_cluster_id'])
            for r in job_candidate_records}
job_beam_not_candidate = [r for r in job_beam_records
                          if (r['run'], r['subrun'], r['event'], r['cluster_id']) not in credited]
RECO_SELECTION_BEAM_NOT_CANDIDATE = f"{RECO_SELECTION_BEAM}_Not{RECO_SELECTION_CANDIDATE}"

bee_set_urls = {}
for population, records in ((RECO_SELECTION_BEAM, job_beam_records),
                            (RECO_SELECTION_CANDIDATE, job_candidate_records),
                            (RECO_SELECTION_BEAM_NOT_CANDIDATE, job_beam_not_candidate)):
    if not MAKE_BEE_SETS:
        print("BEE sets: not built (MAKE_BEE_SETS = False)")
        break
    selections, labels, skipped = OrderedDict(), {}, []
    for r in records:
        rse = (r['run'], r['subrun'], r['event'])
        key = staged_key(r['unit'], *rse)
        if key is None:
            skipped.append(rse)
            continue
        selections.setdefault(key, []).append(
            f"r{rse[0]}_s{rse[1]}_e{rse[2]}_cluster{r['cluster_id']}_{population}")
        labels[key] = rse
    links_path = reco_dir / f"bee_links_{population}.txt"
    if not selections:
        links_path.write_text(f"# BEE SET ({population}): none -- no events with staged JSONs\n")
        print(f"{population}: no events to upload")
        continue
    url, map_path, rows = build_and_upload_set(selections, STAGING_ROOT,
                                               output_dir / f"bee_set_{population}",
                                               f"selected_clusters.txt ({population})")
    bee_set_urls[population] = url
    base = url.rstrip('/').rsplit('/', 1)[0] if url else None
    lines = [f"# BEE SET ({population}, one upload): {url or 'UPLOAD FAILED'}",
             "# run\tsubrun\tevent\tclusters\tstaged\tk\tbee_url"]
    for n, staged, k, figures in rows:
        run, subrun, event = labels[(staged, k)]
        clusters = ",".join(f.split('_cluster')[1].split('_')[0] for f in figures)
        lines.append(f"{run}\t{subrun}\t{event}\t{clusters}\t{staged}\t{k}\t"
                     f"{f'{base}/{n}/' if base else '-'}")
    if skipped:
        lines.append(f"# skipped, no staged JSONs: {sorted(set(skipped))}")
    links_path.write_text("\n".join(lines) + "\n")
    print(f"{population}: {len(rows)} event(s) -> {url}")
    print(f"  per-event links: {links_path}")
    if skipped:
        print(f"  skipped {len(set(skipped))} event(s) with no staged JSONs")